Replay the thing that the edit server does

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pickle
import torch
import matplotlib.pyplot as plt
import tqdm.auto as tqdm

import load_gml
# from gerry13y_edit import Editor
from gerry10_edit import Editor

In [ ]:
editor = Editor()

In [ ]:
folder = Path('data') / 'custom_gmls'
# fname = folder / 'graffiti_1.json'
# fname = folder / 'shapes3.json'
# fname = folder / 'georgia_tech_2.json'
# fname = folder / 'RSS_line_0.json'
fname = folder / 'R_1.json'
fname = folder / 'RSS_2.json'
fname = folder / 'graffiti_0.json'
fname = folder / 'ADAPT_2.json'
drawing = load_gml.Drawing(fname)

In [ ]:
# drawing.strokes = drawing.strokes[:-3]
# drawing.strokes = drawing.strokes[:-3] + drawing.strokes[-2:]
for stroke in drawing.strokes:
    plt.plot(stroke[:, 1], stroke[:, 2])

In [ ]:
import numpy as np
xy = [stroke[:, 1:3] for stroke in drawing.strokes]
dxy = [np.diff(stroke, axis=0) for stroke in xy]
for stroke in dxy:
    plt.plot(stroke)
print([np.linalg.norm(stroke, axis=1).mean() for stroke in dxy])
print(np.nanmean([np.linalg.norm(stroke, axis=1).mean() for stroke in dxy]))

In [ ]:
def weight(x):
    x = torch.from_numpy(x)
    speeds = torch.linalg.norm(torch.diff(x[:, 1:3], axis=0, prepend=x[:1, 1:3]), axis=1)
    # return torch.clamp(speeds * 1e5, 1e-1, 1)
    # return -torch.log10(speeds)
    return -torch.log10(torch.clamp(speeds * 1e1, 1e-1, 1))
    print()
    print(x[:3])
weights = [weight(stroke) for stroke in drawing.strokes]
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for array in drawing.strokes:
    # plt.plot(array[:, 1], array[:, 2], marker='.', )
    print(array.shape, weight(array).shape)
    axes[0].scatter(array[:, 1], array[:, 2], c=weight(array), cmap='viridis')
    axes[1].plot(np.diff(array[:, 1:3], axis=0))
    axes[1].plot(weight(array), 'k:')
axes[0].axis('equal')

In [ ]:
histories = []
def weighted_mse(x, y, weight):
    return (((x - y) ** 2) * weight[None, :, None]).mean()
def loss(weight):
    def loss_(x, y):
        # l1 = torch.nn.MSELoss()(y * weight, x * weight)
        # l2 = torch.nn.MSELoss()(y[:, :, :2] * weight, x[:, :, :2] * weight)
        l1 = weighted_mse(y, x, weight)
        l2 = weighted_mse(y[:, :, :2], x[:, :, :2], weight)
        return 5 * l1 + 10 * l2
    return loss_
def func(array):
    # if array.shape[0] < 15:
    #     return editor.edit([array], guidance_weight=1e5)[0]
    # else:
    #     return editor.edit([array], guidance_weight=1e2, repeat=3,
    #                        t_start=10)[0]
    histories.append([])
    return editor.edit([array], guidance_weight=1e5, repeat=20, t_start=3, history=histories[-1],
    # return editor.edit([array], guidance_weight=1e3, repeat=9, t_start=1, history=histories[-1],
    # return editor.edit([array], guidance_weight=1e2, repeat=20, t_start=1, history=histories[-1],
        # loss=loss(weight(array).to('cuda'))
    )[0]

torch.manual_seed(8675309)
fit = [func(array) for array in drawing.strokes]

histories = [torch.concatenate(history, dim=0).detach().cpu().numpy() for history in histories]

In [ ]:
import style.diffusion_policy_gml.utils as utils
t = histories[0][-1]
# utils.plot_result(editor.dataset, t[:, 2:], t[:, :2]);
def plot_x(ax, x, xorig):
    action = editor.dataset.unnormalize_action(x[:, 2:])
    # xy = editor.dataset.unnormalize_obs(x[:, :2])
    # utils.plot_traj(ax, action, x0=xy[0, :2])
    xorig_ = editor.dataset.unnormalize_obs(xorig[:, 1:3])
    txy = editor.batched_to_strokes(x, editor.dataset, None, xorig_)
    # print(txy[0].shape)
    # print(txy)
    utils.plot_traj(ax, action, obs=txy[0][:, :2])
    # raise Exception

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(10 * 2, 5))
N = histories[0].shape[0]
inds = np.linspace(0, N-1, 5, endpoint=True).astype(int)
print(inds)

shapes = [stroke.shape for stroke in drawing.strokes]
for i, ax in enumerate(axes.flat):
    histi = inds[i]
    for orig, shape, history in zip(drawing.strokes, shapes, histories):
        # ax.plot(history[inds, :, 1], history[inds, :, 2], alpha=0.5)
        plot_x(ax, history[histi], orig)
    ax.set_title(f'$u={histi}$', fontsize=24)
# for i, (ax, array) in enumerate(zip(axes.flat, fit)):
#     ax.plot(array[:, 1], array[:, 2])
#     ax.set_title(i)
fig.suptitle('Evolution of Trajectory over Successive Small Edits', fontsize=36, y=1)
fig.tight_layout()
fig.subplots_adjust(top=0.75)

if False:
    base = fname.with_suffix('').name
    fig.savefig(f'results/gerry13y_edit/{base}_evolution.svg')
    fig.savefig(f'results/gerry13y_edit/{base}_evolution.eps')
    pickle.dump(dict(histories=histories, fit=fit),
                open(Path('results/gerry13y_edit/') / f'{base}_evolution.pkl', 'wb'))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10 * 2, 5 * 2), sharey='row')
N = histories[0].shape[0]
inds = np.linspace(0, N-1, 5, endpoint=True).astype(int)
print(inds)

shapes = [stroke.shape for stroke in drawing.strokes]
# for i, ax in enumerate(axes.flat):
for i, (ax1, ax2) in enumerate(axes.T):
    histi = inds[i]
    for shape, history in zip(shapes, histories):
        # ax.plot(history[inds, :, 1], history[inds, :, 2], alpha=0.5)
        # plot_x(ax, history[histi][:shape[0]])
        speed = np.linalg.norm(hist[histi][:, 2:4], axis=1)
        acc = np.diff(hist[histi][:, 2:4], axis=0)
        acc = np.linalg.norm(acc, axis=1)
        ax1.plot(speed)
        ax2.plot(acc)
    ax1.set_title(f'$u={histi}$', fontsize=24)
# for i, (ax, array) in enumerate(zip(axes.flat, fit)):
#     ax.plot(array[:, 1], array[:, 2])
#     ax.set_title(i)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for array in drawing.strokes:
    axes[0].plot(array[:, 1], array[:, 2], marker='.')
for array in fit:
    print(array.shape)
    axes[1].plot(array[:, 0], array[:, 1])
# for hist in histories:
#     array = hist[-1, :, :]
#     plot_x(axes[1], array)
axes[0].axis('equal')
axes[1].axis('equal')

base = fname.with_suffix('').name
pickle.dump(fit, open(Path('results/gerry13y_edit/') / f'{base}.pkl', 'wb'))

# COMPARE TO ONE_STEP

In [ ]:
results = {}
for t_start in [0, 15, 30, 45, 60]:
    def func(array):
        return editor.edit([array], guidance_weight=1e2, repeat=1, t_start=t_start,
            # loss=loss(weight(array).to('cuda'))
            print_progress=False
        )[0]
        return editor.edit([array], guidance_weight=0, repeat=1, t_start=t_start,
            # loss=loss(weight(array).to('cuda'))
            # loss = lambda *_: [0]
        )[0]

    torch.manual_seed(8675309)
    fit_onestep = [func(array) for array in drawing.strokes]

    results[t_start] = fit_onestep

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(10 * 2, 5))

shapes = [stroke.shape for stroke in drawing.strokes]
for i, (ax, (k, txy)) in enumerate(zip(axes.flat, sorted(results.items()))):
    # for orig, shape, history in zip(drawing.strokes, shapes, histories):
    #     # ax.plot(history[inds, :, 1], history[inds, :, 2], alpha=0.5)
    #     plot_x(ax, history[histi], orig)
    for txy_ in txy:
        utils.plot_traj(ax, txy_, obs=txy_[:, :2])
    ax.set_title(f"$T'={k}$", fontsize=24)
# for i, (ax, array) in enumerate(zip(axes.flat, fit)):
#     ax.plot(array[:, 1], array[:, 2])
#     ax.set_title(i)
fig.suptitle("Editing with $U=1$ for Various Noise Levels $T'$", fontsize=36, y=1)
fig.tight_layout()
fig.subplots_adjust(top=0.75)

if False:
    base = fname.with_suffix('').name
    fig.savefig(f'results/gerry13y_edit/{base}_onestep_true.svg')
    fig.savefig(f'results/gerry13y_edit/{base}_onestep_true.eps')
    pickle.dump(dict(results=results),
                open(Path('results/gerry13y_edit/') / f'{base}_onestep_true.pkl', 'wb'))

In [ ]:
for stroke in drawing.strokes:
    print(stroke.shape, np.prod(stroke.shape))

# Compare effect of guidance

In [ ]:
sse = torch.nn.MSELoss(reduction='sum')
def loss(eta, eta_delta):
    def loss_(x, y):
        l2 = sse(y[:, :, :2], x[:, :, :2])
        l1 = sse(y[:, :, 2:4], x[:, :, 2:4])
        return eta * (l2 + eta_delta * l1)
    return loss_

etas = np.array([1, 10, 100]) * 10
# eta_deltas = np.array([0.25, 0.5, 1, 2, 4])
eta_deltas = np.array([-0.25, 0.25, 1, 4])
to_test = [(eta, eta_delta) for eta in etas for eta_delta in eta_deltas]

results = {}
for eta, eta_delta in tqdm.tqdm(to_test):
    def func(array):
        return editor.edit([array], guidance_weight=1, repeat=20, t_start=1,
            loss=loss(eta, eta_delta),
            print_progress=False
        )[0]

    torch.manual_seed(8675309)
    fit_eta = [func(array) for array in drawing.strokes]

    results[(eta, eta_delta)] = fit_eta

orig = [editor.edit([array], guidance_weight=1, repeat=0, t_start=1,
            print_progress=False
        )[0] for array in drawing.strokes]

In [ ]:
fig, axes = plt.subplots(len(etas), len(eta_deltas) + 1, figsize=(15, 11))

for eta, row in zip(etas, axes):
    for eta_delta, ax in zip(eta_deltas, row[1:]):
        fit_eta = results[(eta, eta_delta)]
        for stroke in fit_eta:
            utils.plot_traj(ax, stroke, obs=stroke[:, :2])
        ax.set_title(f"$\eta={eta},~~\eta_\Delta={eta_delta}$", fontsize=24)

for stroke in orig:
    utils.plot_traj(axes[1][0], stroke, obs=stroke[:, :2])
axes[1][0].set_title("Input Trajectory", fontsize=24)

axes[0][0].axis('off')
axes[2][0].axis('off');

fig.suptitle("Effect of ``Similarity Guidance'' Strength on Adherence to Input Trajectory", fontsize=36, y=1)
fig.tight_layout()

if False:
    base = fname.with_suffix('').name
    fig.savefig(f'results/gerry13y_edit/{base}_eta.svg')
    fig.savefig(f'results/gerry13y_edit/{base}_eta.eps')
    pickle.dump(dict(results=results, orig=orig),
                open(Path('results/gerry13y_edit/') / f'{base}_eta.pkl', 'wb'))